<a href="https://colab.research.google.com/github/aKi-1201/breeze-asr-26/blob/main/Breeze_ASR_26_%E5%8F%B0%E8%AA%9E%E8%AA%9E%E9%9F%B3%E8%BE%A8%E8%AD%98%E6%B8%AC%E8%A9%A6_v_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Breeze-ASR-26 台語語音辨識測試 v.1

* 改用 Breeze-ASR-26 (台語) 模型進行辨識。
* 可使用 .txt 檔案放要辨識的清單。
* 可自訂字幕檔要儲存的資料夾。
* 可自訂是否要跳過已辨識過的。
* 改用 yt-dlp 下載影音檔案。

### ▋ 簡介
Breeze-ASR-26 是 MediaTek Research 針對台語 (Taigi) 的自動語音辨識模型（基於 Whisper 微調），模型與說明可參考:
* https://huggingface.co/MediaTek-Research/Breeze-ASR-26

結合 Breeze-ASR-26 和 yt-dlp 的工具，就可以將 Youtube 上的影片或播放清單擷取聲音、儲存語音檔後，進行語音辨識，並生成字幕檔。

目前在後面程式設定區塊中，語音來源路徑的「url」欄位中，可以填入 Youtube 的影片或影片清單網址；如果是在電腦本機中的影片或聲音檔，則可以上載在左側資料夾中後，在「url」欄位中填入完整的檔名名稱。

接著將其它選項都設定好後，就可以在[程式區塊]中按「執行」的按鈕，開始進行語音辨識了。

Breeze-ASR-26 專為台語設計，輸出為中文漢字。若想用自動偵測或其他語言代碼，亦可自行調整設定。

程式第一次執行時，因為要安裝及下載自動語音辨識所需要的資料，可能要稍等一下下。

In [ ]:
#@title 掛載雲端硬碟 { vertical-output: true }
#@markdown <font size="6">👈</font> <b>請按左側執行鈕，再依提示選帳號及授權，即可將雲端硬碟掛載在 /content/drive</b>
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Breeze-ASR-26 台語語音辨識並輸出字幕檔案程式 { vertical-output: true }

#@markdown <font size="6">👈</font> <b>設定底下的自訂參數後，就可以按左側的執行鈕</b>

#@markdown ▋ <b>聲音檔的來源</b>，可以是網址(Youtube影片、撥放清單、 [Vocaroo](https://voca.roo/) 網址)，或是上載後的影片、聲音檔檔案名稱。<br />有多個可以放入 .txt 檔案中(例: list.txt)，一行一個網址或是檔案路徑。
url = "/content/drive/MyDrive/test_audio.mp3" #@param {type:"string"}
#@markdown ▋ <b>語音的語言代碼</b> (Breeze-ASR-26 為台語模型，輸出為中文漢字；可用自動判斷或輸入語言代碼)
lang = "Chinese" # @param ["Chinese", "English", "Japanese", "Korean", "\u81EA\u52D5\u5224\u65B7"] {allow-input: true}
#markdown ▋ <b>輸出為哪一種格式</b>（.srt:字幕檔、.txt:純文字檔、.srt.txt:字幕檔但改了檔名）
outputFormat = 'txt' #param ['srt', 'txt', '.srt.txt']
#@markdown ▋ <b>輸出到哪個資料夾</b> (空白表示使用預設值)
output_path = '' #@param {type:"string"}
#@markdown ▋ <b>是否自動建立資料夾</b> (有設定輸出的資料夾時才有用)
makedirs_enable = True #@param { type: 'boolean' }
#@markdown ▋ <b>是否覆蓋已存在的辨識結果</b> (沒勾選會跳過已辨識過的)
overwrite_enable = True #@param { type: 'boolean' }
model_id = "MediaTek-Research/Breeze-ASR-26"
#@markdown ▋ <b>長音檔分段秒數</b>（建議 20~30，0 表示不分段）
chunk_length_s = 30 #@param {type:"number"}
#@markdown ▋ <b>分段重疊秒數</b>（避免斷句）
stride_length_s = 5 #@param {type:"number"}
#@markdown ▋ <b>是否全部辨識完成，立即下載字幕檔</b>
start_downloading_immediately = True #@param { type: 'boolean' }
#@markdown ▋ <b>是否即時顯示語音辨識結果</b>
verbose = False #@param { type: 'boolean' }
#@markdown ▋ <b>是否顯示安裝過程</b> (除錯用)
debug = False #@param { type: 'boolean' }
#@markdown ---
#@markdown <center><p><font color="blue">以下為指令輸出內容</p></center>


# Install + Import + Config
try: from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline
except:
  print('請稍候, 安裝相關工具, 會花一點時間...')
  print('>> install transformers ...')
  if debug :
    ! pip -q install transformers accelerate torchaudio
  else :
    ! pip -q install transformers accelerate torchaudio > /dev/null 2>&1

try: from yt_dlp import YoutubeDL
except :
  print('>> install yt_dlp ...')
  if debug :
    ! pip -q install yt_dlp
  else :
    ! pip -q install yt_dlp  > /dev/null 2>&1

try: from slugify import slugify
except:
  print('>> install python-slugify ...')
  if debug :
    ! pip install python-slugify
  else :
    ! pip install python-slugify  > /dev/null 2>&1

print('\n')

import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline
from yt_dlp import YoutubeDL
import re
import os
import urllib.request
from slugify import slugify

import google

#url = "https://voca.ro/15HVH0YvIaa6"
#url = 'https://www.youtube.com/watch?v=I4DZn4z8aRQ&list=PLelNvYGEtsV8TpwxL4t7GTTG-7qALZqol'
#url = 'https://www.youtube.com/watch?v=I4DZn4z8aRQ'
#url = 'vocaroo-\u53F0\u8A9E.mp3'

#lang = 'Chinese' # '\u81EA\u52D5\u5224\u65B7'
#start_downloading_immediately = True

#model_id = 'MediaTek-Research/Breeze-ASR-26'
#outputFormat = 'txt' # 'srt' 'txt'

#如果是 .srt.txt 會先以字幕格式儲存，但將檔名後面再加上 .txt
renameSrtToTxt = False
if outputFormat == '.srt.txt' :
  outputFormat = 'srt'
  renameSrtToTxt = True

audioFile = 'source.mp3' #暫存的語音檔檔名
tempFolder = '.' #暫存的資料夾(工作目錄、下載的影音、剛轉好的文字檔)
#output_path = '.'
title = ''
textFileList = [] #記錄轉好的文字檔檔名清單

LANG_CODE_MAP = {
  'Chinese': 'zh',
  'English': 'en',
  'Japanese': 'ja',
  'Korean': 'ko'
}

#檢查輸出文字檔的資料夾是否存在, 不存在就改用預設值
if re.sub(r'\s', '', output_path) == '' :
  #如果沒有輸入，就輸出到預設的暫存資料夾
  output_path = tempFolder
else :
  if os.path.exists(output_path) :
    if not os.path.isdir(output_path) :
      print(output_path + ' 不是資料夾哦! 改用預設值: '+tempFolder)
      output_path = tempFolder
  elif makedirs_enable :
    print('指定的輸出目錄 '+output_path+' 不存在,\n>>>> 新增: '+output_path)
    os.makedirs(output_path)
  else :
    print('指定的輸出目錄 '+output_path+' 不存在, 改用預設值: '+tempFolder)
    output_path = tempFolder


# GPU or CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_ID = 0 if torch.cuda.is_available() else -1
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
ASR_PIPELINE = None
ASR_PROCESSOR = None

def get_language_code() :
  if lang=="\u81EA\u52D5\u5224\u65B7" :
    return None
  return LANG_CODE_MAP.get(lang, lang)

def get_asr_pipeline() :
  global ASR_PIPELINE
  global ASR_PROCESSOR
  if ASR_PIPELINE is None :
    print('\n>>>>> \u8F09\u5165\u6A21\u578B...')
    ASR_PROCESSOR = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
      model_id,
      torch_dtype=TORCH_DTYPE,
      low_cpu_mem_usage=True,
      use_safetensors=True
    )
    ASR_PIPELINE = pipeline(
      "automatic-speech-recognition",
      model=model,
      tokenizer=ASR_PROCESSOR.tokenizer,
      feature_extractor=ASR_PROCESSOR.feature_extractor,
      device=DEVICE_ID
    )
  return ASR_PIPELINE

def build_generate_kwargs(processor) :
  kwargs = {
    'task': 'transcribe'
  }
  language = get_language_code()
  if language :
    kwargs.update({'language': language})
  return kwargs

def get_return_timestamps(fileType) :
  if fileType == 'txt' :
    return None
  return True

def build_chunking_args() :
  args = {}
  if isinstance(chunk_length_s, (int, float)) and chunk_length_s > 0 :
    args['chunk_length_s'] = float(chunk_length_s)
    if isinstance(stride_length_s, (int, float)) and stride_length_s > 0 :
      args['stride_length_s'] = float(stride_length_s)
  return args

def format_timestamp(seconds) :
  if seconds is None :
    seconds = 0.0
  seconds = max(0.0, float(seconds))
  whole = int(seconds)
  millis = int(round((seconds - whole) * 1000))
  if millis == 1000 :
    whole += 1
    millis = 0
  hours = int(whole // 3600)
  minutes = int((whole % 3600) // 60)
  secs = int(whole % 60)
  return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

def normalize_chunks(result) :
  chunks = result.get('chunks', []) if isinstance(result, dict) else []
  normalized = []
  for chunk in chunks :
    if not isinstance(chunk, dict) :
      continue
    ts = chunk.get('timestamp')
    if ts is None :
      ts = chunk.get('timestamps')
    if ts is None and 'start' in chunk and 'end' in chunk :
      ts = (chunk['start'], chunk['end'])
    if ts is None or len(ts) != 2 :
      continue
    start, end = ts
    if start is None or end is None :
      continue
    text = chunk.get('text', '')
    normalized.append((float(start), float(end), text))
  return normalized

def write_srt(cues, output_file) :
  with open(output_file, 'w', encoding='utf-8') as f :
    index = 1
    for start, end, text in cues :
      if text is None :
        continue
      safe_text = str(text).strip()
      if safe_text == '' :
        continue
      f.write(f"{index}\n")
      f.write(f"{format_timestamp(start)} --> {format_timestamp(end)}\n")
      f.write(f"{safe_text}\n\n")
      index += 1

def build_srt_cues(result) :
  chunks = normalize_chunks(result)
  if not chunks :
    return []
  cues = []
  for start, end, text in chunks :
    text = str(text).strip()
    if text != '' :
      cues.append((start, end, text))
  return cues


def getYoutubePlaylistInfo_ydl(url) :
  """
  使用 yt_dlp 由 Youtbue 播放清單網址擷取各影片的資訊
  :param url: Youtube 影片播放清單網址
  """
  ydl_opts = {
      'no_warnings':True,
      'quiet':True,
      #'extract_flat':True
  }
  with YoutubeDL(ydl_opts) as ydl:
    info_dict = ydl.extract_info(url, download=False)
    #return (info_dict.get('entries', None)) #entries 為清單中的影片, 'webpage_url' 為影片網址
    return info_dict

def getYoutubeTitle_ydl(url) :
  """
  使用 yt_dlp 由 Youtbue 網址擷取影片的標題字
  :param url: Youtube 影片網址
  """
  ydl_opts = {
      'no_warnings':True,
      'quiet':True
  }
  with YoutubeDL(ydl_opts) as ydl:
    info_dict = ydl.extract_info(url, download=False)
    return(info_dict.get('title', None))

def getAudioFromYoutube_ydl(url, output_path, outputFilename):
  """
  使用 yt_dlp 由 Youtbue 網址下載影片的聲音並存檔
  :param url: Youtube 影片網址
  :param output_path: 儲存的資料夾名稱
  :param outputFilename: 聲音檔的檔名
  """
  #filename為去掉副檔名的路徑，以免變成 xxx.mp3.mp3
  filename = os.path.join(output_path, os.path.splitext(outputFilename)[0])
  ydl_opts = {
      'no_warnings':True,
      'quiet':True,
      'outtmpl':filename,
      'keepvideo':False,
      'format': 'bestaudio/best',
      'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
      #  'preferredquality': '192',
      }],
      'overwrites':True
  }
  with YoutubeDL(ydl_opts) as ydl:
    ydl.download(url)

#def getAudioFromYoutube(url, output_path, filename) :
#  """
#  使用 pytube, 由 Youtbue 網址下載影片的聲音並存檔
#  :param url: Youtube 影片網址
#  :param output_path: 儲存的資料夾名稱
#  :param filename: 聲音檔的檔名
#  """
#  video = YouTube(url)
#  if not YoutubeDL_enable :
#    video.streams.get_audio_only().download(output_path, filename)
#  else :
#    getAudioFromYoutube_ydl(url, output_path, filename)
#  return video.title
#


def getVocarooMP3URL(url) :
  """
  Get the MP3 URL from Vocaroo record share URL
  :param url: Vocaroo url
  """
  vocarooMP3Base = 'https://media.vocaroo.com/mp3/'
  regex = re.compile(r'(https\:\/\/voca\.ro|https\:\/\/vocaroo\.com)\/(\w{12})')
  match = regex.search(url)
  if match :
    url = vocarooMP3Base+match.group(2)
  return url

def transcribe(audioFilename, output_path, outputFilename, outputFormat) :
  """
  load Breeze-ASR-26 model and get transcribe result
  :param audioFilename: audio file to transcribe
  :param output_path: the folder to save result
  :param outputFilename: the filename(without the extension) to save result
  :param outputFormat: the transcribe result format (thie extension of filename)
  """
  #print(audioFilename+' --> '+output_path+outputFilename+'.'+outputFormat)
  #return 1
  asr = get_asr_pipeline()
  print('\n>>> \u958B\u59CB\u89E3\u8B6F......')
  generate_kwargs = build_generate_kwargs(ASR_PROCESSOR)
  asr_kwargs = {
    'return_timestamps': get_return_timestamps(outputFormat),
    'generate_kwargs': generate_kwargs
  }
  asr_kwargs.update(build_chunking_args())
  asr_kwargs = {k: v for k, v in asr_kwargs.items() if v is not None}
  result = asr(audioFilename, **asr_kwargs)

  if verbose and isinstance(result, dict) :
    print(result.get('text', ''))

  # save result to outputFormat file
  saveToFile(result, output_path, outputFilename, outputFormat)

  #額外處理 .srt.txt (2024.11.22 add)
  if renameSrtToTxt :
    pathSrtTxt = getOutputTextFilename(outputFilename)
    pathSrt = re.sub(r'\.txt', '', pathSrtTxt)
    # saveToFile 會存成 .srt, 將它更名為 .srt.txt
    os.rename(pathSrt, pathSrtTxt)

def saveToFile(result, output_path, filename, fileType='srt') :
  """
  save ASR transcribe result to a file
  :param result: transcribe result
  :param output_path: the folder to save result
  :param filename: the filename(without the extension) to save result
  :param fileType: the transcribe result format (thie extension of filename)
  """
  if fileType == 'srt' :
    outputFile = getOutputSrtFilename(filename) if renameSrtToTxt else getOutputTextFilename(filename)
  else :
    outputFile = getOutputTextFilename(filename)
  if fileType == 'txt' :
    text = result.get('text', '') if isinstance(result, dict) else ''
    with open(outputFile, 'w', encoding='utf-8') as f :
      f.write(str(text).strip())
    return
  if fileType != 'srt' :
    print('>>> \u4E0D\u652F\u63F4\u6B64\u683C\u5F0F, \u6539\u70BA\u8F38\u51FA\u7D14\u6587\u5B57\u6A94')
    text = result.get('text', '') if isinstance(result, dict) else ''
    with open(outputFile, 'w', encoding='utf-8') as f :
      f.write(str(text).strip())
    return

  cues = build_srt_cues(result)
  if not cues :
    text = result.get('text', '') if isinstance(result, dict) else ''
    if str(text).strip() == '' :
      print('>>> \u7121\u8B58\u5225\u7D50\u679C\u53EF\u5B58\u6A94')
      return
    cues = [(0.0, 0.0, str(text).strip())]
  write_srt(cues, outputFile)


#
def getFilenameFromTitle(title) :
  """
  convert title to valid filename
  :param title: text to slugify
  """
  return slugify(title, allow_unicode=True, lowercase=False)

def getOutputSrtFilename(filename) :
  fPath = os.path.relpath(output_path)
  return os.path.join(fPath, filename+'.srt')

def getOutputTextFilename(filename) :
  fPath = os.path.relpath(output_path)
  #如果是 .srt.txt 會先以字幕格式儲存，但將檔名後面再加上 .txt
  if renameSrtToTxt :
    return os.path.join(fPath, filename+'.' + outputFormat + '.txt')
  else :
    return os.path.join(fPath, filename+'.'+outputFormat)


def isOutputTextFileExists(filename) :
  return os.path.exists(getOutputTextFilename(filename))

def removeAudioFile(filename) :
  if os.path.exists(filename) :
    os.remove(filename)

def convertNewLineFormat(filename) :
  if os.path.exists(filename) :
    f = open(filename, 'r+')
    data = f.read()
    data = re.sub('\r', '', data, flags=re.MULTILINE) #remove all \r
    data = re.sub('\n', '\r\n', data, flags=re.MULTILINE) #replace \n to \r\n
    f.seek(0)
    f.write(data)
    f.truncate()
    f.close()

def getAudioAndTranscribe(urlOrFile) :
  filename = audioFile #語音檔檔名(辨識用的暫存檔，用固定的名稱較方便)
  # 分網址或是Colab儲存空間檔案處理
  if re.search(r'https\:\/\/', urlOrFile) :
    isPlayList = False
    # Youtube 分播放清單跟單一影片處理
    if re.search(r'youtube\.|youtu\.', urlOrFile) :
      # 建立 Playlist 物件
      #pList = Playlist(urlOrFile)
      videoInfo = getYoutubePlaylistInfo_ydl(urlOrFile)
      isPlayList = True
      #try : title = pList.title
      try : videos = videoInfo['entries']
      except :
        isPlayList = False
      #處理 Youtube 播放清單
      if isPlayList :
        title = videoInfo['title']
        print('\n>>\u8655\u7406 Youtube \u64AD\u653E\u6E05\u55AE\u5F71\u7247: '+title)
        #for video in pList.videos :
        for video in videos :
          #title = video.title #取得Youtube影片的標題字
          title = video['title'] #取得Youtube影片的標題字
          #filename = title+'.mp3'
          print('\n>>> '+title)
          #用影片的標題當主檔名，但需將不適合的字置換或去除
          #convert title to valid filename
          outputFilename = getFilenameFromTitle(title)
          #continue

          downloadOk = True
          #如果設定辨識過的重新辨識(overwrite_enable), 或是輸出字幕檔不存在就呼叫
          if not isOutputTextFileExists(outputFilename) or overwrite_enable :
            # delete the old audio file
            removeAudioFile(filename)
            # download audio from video stream
            #video.streams.get_audio_only().download(tempFolder, filename)
            print('>> \u4E0B\u8F09\u5F71\u97F3\u6A94\u6848\u4E2D...')
            try: getAudioFromYoutube_ydl(video['webpage_url'], tempFolder, filename)
            except:
              print('>>>\u7121\u6CD5\u7531\u7DB2\u5740\u4E0B\u8F09\u5F71\u97F3')

            #print('\n>> \u6E96\u5099\u9032\u884C\u8A9E\u97F3\u8FA8\u8B58...\n')
            if os.path.exists(filename) :
              print('>>>Breeze-ASR-26 \u8FA8\u8B58\u4E2D...\n')
              # load model and get transcribe result
              transcribe(filename, output_path, outputFilename, outputFormat)
            else :
              downloadOk = False
              print('>>>\u627E\u4E0D\u5230\u8A9E\u97F3\u6A94\uFF0C\u8ACB\u78BA\u5B9A\u5F71\u97F3\u6A94\u662F\u5426\u5DF2\u6E96\u5099\u597D\u4E86\n')

          #將輸出的檔名列入結果清單中，方便後續壓縮、下載
          if downloadOk:
            textFileList.append(getOutputTextFilename(outputFilename))
        #title = pList.title #供製作下載壓縮檔用的主檔名
        title = videoInfo['title']
      else :
        # Youtube single video
        #title = YouTube(urlOrFile).title #取得Youtube影片的標題字
        title = videoInfo['title']
        outputFilename = getFilenameFromTitle(title) #用影片的標題當主檔名，但需將不適合的字置換或去除
        downloadOk = True
        #如果設定辨識過的重新辨識(overwrite_enable), 或是輸出字幕檔不存在就呼叫
        if not isOutputTextFileExists(outputFilename) or overwrite_enable :
          # delete the old audio file
          removeAudioFile(filename)
          #getAudioFromYoutube(urlOrFile, tempFolder, filename)
          print('>> \u4E0B\u8F09\u5F71\u97F3\u6A94\u6848\u4E2D...')
          try: getAudioFromYoutube_ydl(videoInfo['webpage_url'], tempFolder, filename)
          except:
            downloadOk = False
            print('>>>\u7121\u6CD5\u7531\u7DB2\u5740\u4E0B\u8F09\u5F71\u97F3')
        #將輸出的檔名列入結果清單中，方便後續壓縮、下載
        if downloadOk:
          textFileList.append(getOutputTextFilename(outputFilename))
    else :
      #非Youtube的網址
      # delete the old audio file
      removeAudioFile(filename)
      isPlayList = False
      if re.search(r'drive\.google\.com', urlOrFile) :
        #Google drive files
        title = getYoutubeTitle_ydl(urlOrFile) #取得檔名
        outputFilename = getFilenameFromTitle(title) #主檔名，但需將不適合的字置換或去除
        if not isOutputTextFileExists(outputFilename) or overwrite_enable :
          getAudioFromYoutube_ydl(urlOrFile, tempFolder, filename)
        #將輸出的檔名列入結果清單中，方便後續壓縮、下載
        if isOutputTextFileExists(outputFilename):
          textFileList.append(getOutputTextFilename(outputFilename))
      elif re.search(r'voca\.ro|vocaroo\.com', urlOrFile) :
        # Vocaroo or other web audio
        urlOrFile = getVocarooMP3URL(urlOrFile) #取得儲存 Vocaroo 的語音檔網址
        title = "vocaroo-"+os.path.basename(urlOrFile) #用 Vocaroo 的 id 當標題字
        outputFilename = title #用 vocaroo-xxxxxxxxxxx 當輸出的主檔名
        if not isOutputTextFileExists(outputFilename) or overwrite_enable :
          urllib.request.urlretrieve(urlOrFile, filename)
        #將輸出的檔名列入結果清單中，方便後續壓縮、下載
        if isOutputTextFileExists(outputFilename):
          textFileList.append(getOutputTextFilename(outputFilename))
      else :
        # other website
        #os.remove(filename)
        title = ''
        removeAudioFile(filename)
        downloadOk = True
        print('\n>> \u4E0B\u8F09\u5F71\u97F3\u6A94\u6848\u4E2D...\n')
        try : videoInfo = getYoutubePlaylistInfo_ydl(urlOrFile)
        except:
          print('\n>>> \u7121\u6CD5\u4E0B\u8F09\u8A72\u7DB2\u5740\u7684\u5F71\u97F3\u8CC7\u8A0A ...\n')
        try: title = videoInfo['title'] #取得標題字
        except:
          title = 'unknow'
        outputFilename = getFilenameFromTitle(title) #用影片的標題當主檔名，但需將不適合的字置換或去除
        try:
          getAudioFromYoutube_ydl(videoInfo['webpage_url'], tempFolder, filename)
        except:
          downloadOk = False
          print('\n>>> \u7121\u6CD5\u4E0B\u8F09\u5F71\u97F3\n')
        #將輸出的檔名列入結果清單中，方便後續壓縮、下載
        if downloadOk :
          textFileList.append(getOutputTextFilename(outputFilename))
        #! yt-dlp -q --force-overwrites -x --audio-format mp3 -o {audioFile} {url}
  else :
    #Colab儲存空間檔案處理(非網址的影音檔)
    isPlayList = False
    title = os.path.splitext(os.path.basename(urlOrFile))[0] #去掉資料夾名稱及附檔名，只取主檔名
    outputFilename = getFilenameFromTitle(title) #結果輸出檔的主檔名
    filename = urlOrFile #辨識指定的影音檔
    if os.path.exists(filename) :
      #將輸出的檔名列入結果清單中，方便後續壓縮、下載
      textFileList.append(getOutputTextFilename(outputFilename))
      #如果字幕檔已存在，而且不可覆蓋，就故意將語音檔設為不存在的檔名，這樣就不會重新辨識
      if (not overwrite_enable) and isOutputTextFileExists(outputFilename) :
        filename = 'fakeFilename.fake'
  #辨識非播放清單的單一語音檔
  if not isPlayList :
    print('\n>> '+title)
    #convert title to valid filename
    #outputFilename = slugify(title, allow_unicode=True, lowercase=False)
    #outputFilename = getFilenameFromTitle(title)
    # 語音檔存在的話，就進行辨識
    # load model and get transcribe result
    print('\n>> \u6E96\u5099\u9032\u884C\u8A9E\u97F3\u8FA8\u8B58...\n')
    if os.path.exists(filename) :
      print('>>>Breeze-ASR-26 \u8FA8\u8B58\u4E2D...\n')
      transcribe(filename, output_path, outputFilename, outputFormat)
    else :
      print('>>>\u627E\u4E0D\u5230\u8A9E\u97F3\u6A94\uFF0C\u8ACB\u78BA\u5B9A\u5F71\u97F3\u6A94\u662F\u5426\u5DF2\u6E96\u5099\u597D\u4E86\n')
    #  start_downloading_immediately = False
  return title

#\u89E3\u6790 .txt \u6A94\u6848\u4E2D\u7684\u6E05\u55AE\uFF0C\u4E00\u884C\u884C\u9032\u884C\u53BB\u6293\u97F3\u6A94\u4E26\u9032\u884C\u8FA8\u8B58
def parseTxtFileAndTranscribe(txtFile) :
  if os.path.exists(txtFile) :
    file = open(txtFile, 'r')
    lines = file.readlines()
    file.close()
    #print(len(lines))
    for line in lines :
      line = re.sub('\r|\n', '', line, re.MULTILINE) #\u53BB\u6389\u6240\u6709\u63DB\u884C\u5B57\u5143
      #\u6E05\u55AE\u4E2D\u7684\u6A94\u6848\u4E0D\u5B58\u5728\uFF0C\u800C\u4E14\u975E\u7D55\u5C0D\u5F91
      #\u8A66\u8457\u52A0\u4E0A\u6E05\u55AE\u6A94\u6848\u7684\u76EE\u9304\u540D\u7A31\u524D\u9762
      if (re.search(r'https\:\/\/', line) is None) and (not os.path.isabs(line)) and (not os.path.exists(line)) :
        p = os.path.dirname(txtFile)
        p = os.path.join(p, line)
        if os.path.exists(p) :
          line = p
      print(f'\n\n>>>>>> {line}')
      getAudioAndTranscribe(line)
  else :
    print('\u627E\u4E0D\u5230\u6A94\u6848: '+txtFile)

if (re.search(r'https\:\/\/', url) is None) and (not re.search(r'\.txt$', url, re.IGNORECASE) is None) :
  #\u89E3\u6790 .txt \u6A94\u4E2D\u7684\u6E05\u55AE\u5F8C\uFF0C\u518D\u9032\u884C\u8FA8\u8B58
  parseTxtFileAndTranscribe(url)
  title = os.path.splitext(os.path.basename(url))[0] #\u53BB\u6389\u8CC7\u6599\u593E\u540D\u7A31\u53CA\u9644\u6A94\u540D\uFF0C\u53EA\u53D6\u4E3B\u6A94\u540D
else :
  #\u76F4\u63A5\u8FA8\u8B58 url \u6307\u5B9A\u7684\u7DB2\u5740\u6216\u662F\u6A94\u6848
  title = getAudioAndTranscribe(url)


#print(textFileList)
if len(textFileList)>0 and start_downloading_immediately :
  print('download...')
  #print(textFileList)
  filenames_list = ''
  for textFile in textFileList :
    #print(textFile)
    if os.path.exists(textFile) :
      convertNewLineFormat(textFile) # replace \n with \r\n (Windows format)
      if os.getcwd() != os.path.dirname(os.path.abspath(textFile)) :
        filename = os.path.basename(textFile)
        #copy textFile to current work directory
        ! cp {textFile} ./
      else :
        filename = textFile
      if filenames_list != '' :
        filenames_list = filenames_list+' '
      filenames_list = filenames_list+filename
  #print(filenames_list)

  if len(textFileList)>1 :
    print('\n\u58D3\u7E2E\u4E26\u4E0B\u8F09\u8FA8\u8B58\u7D50\u679C')
    outputFilename = slugify(title, allow_unicode=True, lowercase=False)+'-'+outputFormat+'.zip'
    ! zip {outputFilename} -D {filenames_list}
  else :
    print('\n\u4E0B\u8F09\u8FA8\u8B58\u7D50\u679C')
    outputFilename = filenames_list
  if os.path.exists(outputFilename):
    google.colab.files.download(outputFilename)
  else:
    print('\n>>>>>>> \u7121\u8FA8\u8B58\u7D50\u679C\u53EF\u4E0B\u8F09')

  '''
  if isPlayList :
    print('\n\u58D3\u7E2E\u4E26\u4E0B\u8F09\u8FA8\u8B58\u7D50\u679C')
    #title = pList.title
    outputFilename = slugify(title, allow_unicode=True, lowercase=False)+'-'+outputFormat+'.zip'
    #\u5148\u5C07\u6240\u6709\u5B57\u5E55\u6A94\u7684\u63DB\u884C\u7B26\u865F\u7531 \n \u63DB\u6210 \r\n\uFF0C\u5168\u90E8\u58D3\u7E2E
    ! zip -l {outputFilename} *.{outputFormat}
  else :
    print('\n\u4E0B\u8F09\u8FA8\u8B58\u7D50\u679C')
    outputFilename = outputFilename+'.'+outputFormat
  #\u4E0B\u8F09\u8FA8\u8B58\u7D50\u679C
  google.colab.files.download(outputFilename)
  '''